In [0]:
%sql
select scheduled, count(distinct Document_Id) as report_cnt from dataplatform01_central_prd_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata_prd
group by scheduled
order by report_cnt asc

In [0]:
%sql
select "BO Deliveries" as category, count(distinct document_id) from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted
where BO_delivery=true
union all 
select "Email Deliveries" as category, count(distinct document_id) from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted
where email_delivery=true
union all 
select "File Deliveries" as category, count(distinct document_id) from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted
where file_delivery=true

select distinct email from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted_email as wse1
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_active as wa1
on wse1.document_id=wa1.document_id
where wa1.document_id is not null
and wa1.schedule_activeness ='Active'
and wse1.email is not null

In [0]:
%sql
select 
distinct
  aws1.Document_Id, 
  aws1.schema_Name, 
  aws1.table_Name, 
  cms2.DataSource_Name, 
  cms2.Document_name, 
  cms2.Full_path, 
  cms2.Document_CUID, 
  up3.DisplayName as owner
--   cms2.SQL_Query as SQL_Query
from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source as aws1
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms2
  on aws1.Document_Id = cms2.Document_Id
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as up3
  on cms2.created = up3.UserName
where aws1.Document_Id is not null
  and aws1.table_Name is not null
  and aws1.table_Name != '' 
  and cms2.DataSource_Type = 'fhsql'
-- group by 
--   aws1.Document_Id, 
--   aws1.schema_Name, 
--   aws1.table_Name, 
--   cms2.DataSource_Name, 
--   cms2.Document_name, 
--   cms2.Full_path, 
--   cms2.Document_CUID, 
--   up3.DisplayName
order by aws1.Document_Id asc, aws1.schema_Name asc, aws1.table_Name asc
-- select distinct DataSource_Name
--  from  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
-- where DataSource_Type='unv'


select distinct aws1.table_Name from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source as aws1
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms2
  on aws1.Document_Id = cms2.Document_Id
where aws1.table_Name not in (select distinct BO_SQL_TABLE from dataplatform01_central_dev_catalog_01.custom_sap_bo.table_linage)
and cms2.DataSource_Type = 'fhsql'

In [0]:
%sql 
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_ucflagged_domain as
select *,
  case
    when upper(trim(Calc_source_table)) like 'TBCL%' 
      or upper(trim(Calc_source_table)) like 'TBBU_CLAIMS%' 
      then 'CLAIMS'
    when upper(trim(Calc_source_table)) like 'TBPO%' 
      or upper(trim(Calc_source_table)) like 'TBBU_POLIC%' 
      or upper(trim(Calc_source_table)) like 'TBMI_POLIC%' 
      or upper(trim(Calc_source_table)) like 'TBMT_POLIC%' 
      then 'POLICIES'
    when Calc_source_schema in ('AP','APPLSYS','APPS','AR','CUST','FA','GL','HR','INV','JTF','ORABOFP','PO')
      then 'FINANCE'
    else null
  end as Temp_Domain
from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_ucflagged
-- where 
--   (upper(trim(Calc_source_table)) like 'TBCL%' 
--     or upper(trim(Calc_source_table)) like 'TBBU_CLAIMS%'
--     or upper(trim(Calc_source_table)) like 'TBPO%'
--     or upper(trim(Calc_source_table)) like 'TBBU_POLIC%'
--     or upper(trim(Calc_source_table)) like 'TBMI_POLIC%'
--     or upper(trim(Calc_source_table)) like 'TBMT_POLIC%'
--     or Calc_source_schema in ('AP','APPLSYS','APPS','AR','CUST','FA','GL','HR','INV','JTF','ORABOFP','PO')
--   )
;

In [0]:
%sql
select distinct Calc_source_table from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_ucflagged_domain
where temp_domain is null

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- Data for Chart 1: Unique reports per cluster ---
pdf_cluster = spark.sql("""
  SELECT COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster, COUNT(DISTINCT Document_Id) AS report_count
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_ucflagged_domain
  GROUP BY cluster
  ORDER BY cluster ASC
""").toPandas()

# --- Data for Chart 2 & 3: Reports by domain per cluster ---
pdf = spark.sql("""
  SELECT COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster,
         COALESCE(Temp_Domain, 'UNIDENTIFIED_DOMAIN') AS Temp_Domain,
         COUNT(DISTINCT Document_Id) AS report_count
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_ucflagged_domain
  GROUP BY cluster, Temp_Domain
  ORDER BY cluster ASC
""").toPandas()

# Total unique reports from entire table
total_reports = spark.sql("""
  SELECT COUNT(DISTINCT Document_Id) AS total
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_ucflagged_domain
""").collect()[0]['total']

# --- Chart 1: Unique Reports Count per Cluster ---
clusters1 = pdf_cluster['cluster'].values
report_counts = pdf_cluster['report_count'].values
x1 = np.arange(len(clusters1))

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 18))

bars = ax1.bar(x1, report_counts, color='#336699')
for i, v in enumerate(report_counts):
    ax1.text(i, v + 20, f'{int(v):,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax1.set_xlabel('Cluster')
ax1.set_ylabel('Distinct Report Count')
ax1.set_title('Unique Reports Count per Cluster')
ax1.set_xticks(x1)
ax1.set_xticklabels(clusters1)

# --- Chart 2: Stacked bar by domain (percentage) ---
pivot = pdf.pivot(index='cluster', columns='Temp_Domain', values='report_count').fillna(0)
pivot = pivot.reindex(clusters1).fillna(0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

clusters2 = pivot_pct.index.values
domains = pivot_pct.columns
x2 = np.arange(len(clusters2))

colors = {'CLAIMS': '#336699', 'POLICIES': '#FABE6E', 'FINANCE': '#6BAF6B', 'UNIDENTIFIED_DOMAIN': '#A3A3A3'}

bottom = np.zeros(len(clusters2))
for domain in domains:
    vals = pivot_pct[domain].values
    ax2.bar(x2, vals, bottom=bottom, color=colors.get(domain, '#888888'), label=domain)
    for i, v in enumerate(vals):
        if v > 2:
            ax2.text(i, bottom[i] + v / 2, f'{v:.1f}%', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    bottom += vals

ax2.text(0.98, 0.95, f'Total Unique Reports: {total_reports:,}', transform=ax2.transAxes,
         fontsize=11, fontweight='bold', va='top', ha='right',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='black', alpha=0.8))
ax2.set_xlabel('Cluster')
ax2.set_ylabel('Percentage (%)')
ax2.set_title('ALL Clusters - Report Distribution by Domain (%)')
ax2.set_xticks(x2)
ax2.set_xticklabels(clusters2)
ax2.legend(title='Domain', fontsize=10, loc='upper left')

# --- Chart 3: Bar height = Chart 1 unique counts, colored proportionally by Chart 2 domain % ---
bottom3 = np.zeros(len(clusters1))
for domain in domains:
    pcts = pivot_pct.reindex(clusters1).fillna(0)[domain].values
    vals = report_counts * (pcts / 100)
    ax3.bar(x1, vals, bottom=bottom3, color=colors.get(domain, '#888888'), label=domain)
    for i, (v, p) in enumerate(zip(vals, pcts)):
        if p > 2:
            ax3.text(i, bottom3[i] + v / 2, f'{p:.1f}%', ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    bottom3 += vals

# Show total count on top of each bar (from Chart 1)
for i, total in enumerate(report_counts):
    ax3.text(i, total + 20, f'{int(total):,}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Total unique reports text box - upper center
ax3.text(0.5, 0.95, f'Total Unique Reports: {total_reports:,}', transform=ax3.transAxes,
         fontsize=11, fontweight='bold', va='top', ha='center',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='black', alpha=0.8))
ax3.set_xlabel('Cluster')
ax3.set_ylabel('Distinct Report Count')
ax3.set_title('Unique Reports per Cluster - Domain Breakdown')
ax3.set_xticks(x1)
ax3.set_xticklabels(clusters1)
ax3.legend(title='Domain', fontsize=10, loc='upper left')

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- Data for unique reports per cluster ---
pdf_cluster = spark.sql("""
  SELECT COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster, COUNT(DISTINCT Document_Id) AS report_count
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_ucflagged_domain
  GROUP BY cluster
  ORDER BY cluster ASC
""").toPandas()

# --- Data for domain percentages per cluster ---
pdf = spark.sql("""
  SELECT COALESCE(CAST(CAST(cluster AS INT) AS STRING), 'UNCLUSTERED') AS cluster,
         COALESCE(Temp_Domain, 'UNIDENTIFIED_DOMAIN') AS Temp_Domain,
         COUNT(DISTINCT Document_Id) AS report_count
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_ucflagged_domain
  GROUP BY cluster, Temp_Domain
  ORDER BY cluster ASC
""").toPandas()

# Total unique reports from entire table
total_reports = spark.sql("""
  SELECT COUNT(DISTINCT Document_Id) AS total
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_full_linage_ucflagged_domain
""").collect()[0]['total']

# --- Prepare data ---
clusters1 = pdf_cluster['cluster'].values
report_counts = pdf_cluster['report_count'].values
x1 = np.arange(len(clusters1))

pivot = pdf.pivot(index='cluster', columns='Temp_Domain', values='report_count').fillna(0)
pivot = pivot.reindex(clusters1).fillna(0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

domains = pivot_pct.columns
colors = {'CLAIMS': '#336699', 'POLICIES': '#FABE6E', 'FINANCE': '#6BAF6B', 'UNIDENTIFIED_DOMAIN': '#A3A3A3'}

# --- Chart 3 only: Bar height = unique counts, colored by domain % ---
fig, ax3 = plt.subplots(figsize=(12, 7))

bottom3 = np.zeros(len(clusters1))
for domain in domains:
    pcts = pivot_pct.reindex(clusters1).fillna(0)[domain].values
    vals = report_counts * (pcts / 100)
    ax3.bar(x1, vals, bottom=bottom3, color=colors.get(domain, '#888888'), label=domain)
    for i, (v, p) in enumerate(zip(vals, pcts)):
        if p > 2:
            ax3.text(i, bottom3[i] + v / 2, f'{p:.1f}%', ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    bottom3 += vals

# Show total count on top of each bar
for i, total in enumerate(report_counts):
    ax3.text(i, total + 20, f'{int(total):,}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Total unique reports text box - upper center
ax3.text(0.5, 0.95, f'Total Unique Reports: {total_reports:,}', transform=ax3.transAxes,
         fontsize=11, fontweight='bold', va='top', ha='center',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='black', alpha=0.8))
ax3.set_xlabel('Cluster')
ax3.set_ylabel('Distinct Report Count')
ax3.set_title('Unique Reports per Cluster - Domain Breakdown')
ax3.set_xticks(x1)
ax3.set_xticklabels(clusters1)
ax3.legend(title='Domain', fontsize=10, loc='upper left')

plt.tight_layout()
plt.show()